# 1. Original versus pre-baseline paired comparison

This notebook compares the two fixed 50/50 fusion experiments on the same 544 held-out participants:

- original symmetric-window fixed-fusion experiment;
- stricter pre-baseline-only fixed-fusion experiment.

No model is retrained.

The notebook measures:

- paired differences in ROC AUC, average precision, balanced accuracy, and Brier score;
- participant-level probability and class changes;
- uncertainty changes;
- which participants lost modality availability;
- whether prediction changes are concentrated among participants whose available modalities changed.

## 1.1. Paths and analysis settings

In [ ]:
# ============================================================
# 1. Paths and analysis settings
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)


try:
    from google.colab import drive

    if not Path(
        "/content/drive/MyDrive"
    ).exists():
        drive.mount(
            "/content/drive"
        )

except ImportError:
    pass


PROJECT_ROOT = Path(
    "/content/drive/MyDrive/adni_mri"
)

MODEL_ROOT = (
    PROJECT_ROOT
    / "models"
    / "3mt_tmc_evidential"
)

TASK_NAME = "mci_prognosis"

ORIGINAL_EXPERIMENT = (
    "gated_cmt_fixed_equal_fusion_md050"
)

PREBASELINE_EXPERIMENT = (
    "temporal_prebaseline_fixed_equal_fusion_md050"
)


ORIGINAL_PREDICTIONS_PATH = (
    MODEL_ROOT
    / "experiments"
    / ORIGINAL_EXPERIMENT
    / "cross_validation_aggregation"
    / TASK_NAME
    / "pooled_predictions"
    / "pooled_out_of_fold_predictions_raw.csv"
)

PREBASELINE_PREDICTIONS_PATH = (
    MODEL_ROOT
    / "experiments"
    / PREBASELINE_EXPERIMENT
    / "cross_validation_aggregation"
    / TASK_NAME
    / "pooled_predictions"
    / "pooled_out_of_fold_predictions_raw.csv"
)


ORIGINAL_INPUT_ROOT = (
    MODEL_ROOT
    / "final_task_ready_inputs"
    / TASK_NAME
)

PREBASELINE_INPUT_ROOT = (
    MODEL_ROOT
    / "temporal_leakage_sensitivity"
    / "prebaseline_only"
    / "final_task_ready_inputs"
    / TASK_NAME
)


COMPARISON_NAME = (
    "original_fixed_vs_prebaseline_fixed"
)

COMPARISON_ROOT = (
    MODEL_ROOT
    / "experiment_comparisons"
    / COMPARISON_NAME
)

TABLE_DIR = (
    COMPARISON_ROOT
    / "tables"
)

FIGURE_DIR = (
    COMPARISON_ROOT
    / "figures"
)

PREDICTION_DIR = (
    COMPARISON_ROOT
    / "aligned_predictions"
)

for directory in [
    COMPARISON_ROOT,
    TABLE_DIR,
    FIGURE_DIR,
    PREDICTION_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


CLASSIFICATION_THRESHOLD = 0.50
BOOTSTRAP_ITERATIONS = 2000
BOOTSTRAP_RANDOM_SEED = 42

EXPECTED_PARTICIPANTS = 544
EXPECTED_FOLDS = [
    0,
    1,
    2,
    3,
    4,
]


BRANCH_MASK_COLUMNS = [
    "BRANCH_MASK__DEMOGRAPHICS",
    "BRANCH_MASK__COGNITIVE_FUNCTIONAL",
    "BRANCH_MASK__CSF",
    "BRANCH_MASK__PLASMA",
    "BRANCH_MASK__APOE",
    "BRANCH_MASK__MRI",
]

BRANCH_DISPLAY_NAMES = {
    "BRANCH_MASK__DEMOGRAPHICS":
        "Demographics",

    "BRANCH_MASK__COGNITIVE_FUNCTIONAL":
        "Cognitive/Functional",

    "BRANCH_MASK__CSF":
        "CSF",

    "BRANCH_MASK__PLASMA":
        "Plasma",

    "BRANCH_MASK__APOE":
        "APOE",

    "BRANCH_MASK__MRI":
        "MRI",
}


print("=" * 72)
print("ORIGINAL FIXED VERSUS PRE-BASELINE FIXED")
print("=" * 72)

print(
    f"\nOriginal pooled predictions:\n"
    f"{ORIGINAL_PREDICTIONS_PATH}"
)

print(
    f"\nPre-baseline pooled predictions:\n"
    f"{PREBASELINE_PREDICTIONS_PATH}"
)

print(
    f"\nComparison outputs:\n"
    f"{COMPARISON_ROOT}"
)

## 1.2. Load and align the pooled out-of-fold predictions

In [ ]:
# ============================================================
# 2. Load and align the pooled out-of-fold predictions
# ============================================================

original_predictions = pd.read_csv(
    ORIGINAL_PREDICTIONS_PATH
)

prebaseline_predictions = pd.read_csv(
    PREBASELINE_PREDICTIONS_PATH
)


REQUIRED_PREDICTION_COLUMNS = [
    "FOLD",
    "RID",
    "TARGET",
    "FINAL_P_pMCI",
    "FINAL_UNCERTAINTY",
    "THREE_MT_P_pMCI",
    "THREE_MT_UNCERTAINTY",
    "TMC_P_pMCI",
    "TMC_UNCERTAINTY",
]


for experiment_name, table in [
    (
        "original",
        original_predictions,
    ),
    (
        "prebaseline",
        prebaseline_predictions,
    ),
]:

    missing_columns = [
        column
        for column in REQUIRED_PREDICTION_COLUMNS
        if column not in table.columns
    ]

    if missing_columns:
        raise KeyError(
            f"{experiment_name} prediction file is missing: "
            + ", ".join(
                missing_columns
            )
        )


original_predictions = (
    original_predictions[
        REQUIRED_PREDICTION_COLUMNS
    ]
    .copy()
)

prebaseline_predictions = (
    prebaseline_predictions[
        REQUIRED_PREDICTION_COLUMNS
    ]
    .copy()
)


for table in [
    original_predictions,
    prebaseline_predictions,
]:

    table["FOLD"] = pd.to_numeric(
        table["FOLD"],
        errors="raise",
    ).astype(
        int
    )

    table["RID"] = pd.to_numeric(
        table["RID"],
        errors="raise",
    ).astype(
        int
    )

    table["TARGET"] = pd.to_numeric(
        table["TARGET"],
        errors="raise",
    ).astype(
        int
    )


if len(
    original_predictions
) != EXPECTED_PARTICIPANTS:
    raise ValueError(
        "Original prediction file does not contain "
        f"{EXPECTED_PARTICIPANTS} participants."
    )

if len(
    prebaseline_predictions
) != EXPECTED_PARTICIPANTS:
    raise ValueError(
        "Pre-baseline prediction file does not contain "
        f"{EXPECTED_PARTICIPANTS} participants."
    )


for experiment_name, table in [
    (
        "original",
        original_predictions,
    ),
    (
        "prebaseline",
        prebaseline_predictions,
    ),
]:

    if table.duplicated(
        subset=[
            "FOLD",
            "RID",
        ]
    ).any():
        raise ValueError(
            f"{experiment_name} predictions contain "
            "repeated FOLD-RID pairs."
        )


paired_predictions = (
    original_predictions.merge(
        prebaseline_predictions,
        on=[
            "FOLD",
            "RID",
        ],
        how="inner",
        suffixes=(
            "_ORIGINAL",
            "_PREBASELINE",
        ),
        validate="one_to_one",
    )
    .sort_values(
        [
            "FOLD",
            "RID",
        ]
    )
    .reset_index(
        drop=True
    )
)


if len(
    paired_predictions
) != EXPECTED_PARTICIPANTS:
    raise ValueError(
        "The experiments do not contain the same "
        f"{EXPECTED_PARTICIPANTS} held-out participants."
    )


if not np.array_equal(
    paired_predictions[
        "TARGET_ORIGINAL"
    ].to_numpy(),
    paired_predictions[
        "TARGET_PREBASELINE"
    ].to_numpy(),
):
    raise ValueError(
        "Targets differ between the two experiments."
    )


paired_predictions[
    "TARGET"
] = paired_predictions[
    "TARGET_ORIGINAL"
].astype(
    int
)


ALIGNED_PREDICTIONS_PATH = (
    PREDICTION_DIR
    / "paired_original_vs_prebaseline_predictions.csv"
)

paired_predictions.to_csv(
    ALIGNED_PREDICTIONS_PATH,
    index=False,
)


print("=" * 72)
print("PAIRED PREDICTION ALIGNMENT")
print("=" * 72)

print(
    f"\nMatched participants: "
    f"{len(paired_predictions)}"
)

print(
    f"sMCI participants: "
    f"{int((paired_predictions['TARGET'] == 0).sum())}"
)

print(
    f"pMCI participants: "
    f"{int((paired_predictions['TARGET'] == 1).sum())}"
)

print(
    "\nMatched folds: "
    + ", ".join(
        str(value)
        for value in sorted(
            paired_predictions[
                "FOLD"
            ].unique()
        )
    )
)

display(
    paired_predictions.head()
)

## 1.3. Metric definitions

In [ ]:
# ============================================================
# 3. Metric definitions
# ============================================================

def calculate_metric(
    targets,
    probabilities,
    metric_name,
):
    targets = np.asarray(
        targets,
        dtype=int,
    )

    probabilities = np.asarray(
        probabilities,
        dtype=float,
    )


    if metric_name == "ROC_AUC":
        return float(
            roc_auc_score(
                targets,
                probabilities,
            )
        )


    if metric_name == "AVERAGE_PRECISION":
        return float(
            average_precision_score(
                targets,
                probabilities,
            )
        )


    if metric_name == "BALANCED_ACCURACY":
        predicted_classes = (
            probabilities
            >= CLASSIFICATION_THRESHOLD
        ).astype(
            int
        )

        return float(
            balanced_accuracy_score(
                targets,
                predicted_classes,
            )
        )


    if metric_name == "BRIER_SCORE":
        return float(
            brier_score_loss(
                targets,
                probabilities,
            )
        )


    raise ValueError(
        f"Unsupported metric: {metric_name}"
    )


METRIC_NAMES = [
    "ROC_AUC",
    "AVERAGE_PRECISION",
    "BALANCED_ACCURACY",
    "BRIER_SCORE",
]


OUTPUT_COLUMNS = {
    "Hybrid": {
        "original":
            "FINAL_P_pMCI_ORIGINAL",

        "prebaseline":
            "FINAL_P_pMCI_PREBASELINE",
    },

    "3MT-only": {
        "original":
            "THREE_MT_P_pMCI_ORIGINAL",

        "prebaseline":
            "THREE_MT_P_pMCI_PREBASELINE",
    },

    "TMC-only": {
        "original":
            "TMC_P_pMCI_ORIGINAL",

        "prebaseline":
            "TMC_P_pMCI_PREBASELINE",
    },
}

## 1.4. Pooled point estimates

In [ ]:
# ============================================================
# 4. Pooled point estimates
# ============================================================

targets = paired_predictions[
    "TARGET"
].to_numpy(
    dtype=int
)

pooled_rows = []


for output_name, columns in (
    OUTPUT_COLUMNS.items()
):

    original_probabilities = paired_predictions[
        columns[
            "original"
        ]
    ].to_numpy(
        dtype=float
    )

    prebaseline_probabilities = paired_predictions[
        columns[
            "prebaseline"
        ]
    ].to_numpy(
        dtype=float
    )


    for metric_name in METRIC_NAMES:

        original_value = calculate_metric(
            targets,
            original_probabilities,
            metric_name,
        )

        prebaseline_value = calculate_metric(
            targets,
            prebaseline_probabilities,
            metric_name,
        )


        pooled_rows.append(
            {
                "OUTPUT":
                    output_name,

                "METRIC":
                    metric_name,

                "ORIGINAL":
                    original_value,

                "PREBASELINE":
                    prebaseline_value,

                "PREBASELINE_MINUS_ORIGINAL":
                    (
                        prebaseline_value
                        - original_value
                    ),
            }
        )


pooled_point_estimates = pd.DataFrame(
    pooled_rows
)


POOLED_POINT_ESTIMATES_PATH = (
    TABLE_DIR
    / "pooled_point_estimates.csv"
)

pooled_point_estimates.to_csv(
    POOLED_POINT_ESTIMATES_PATH,
    index=False,
)


display(
    pooled_point_estimates.round(
        6
    )
)

## 1.5. Paired bootstrap comparison

The same resampled participant indices are used for both experiments. This preserves the paired design and directly estimates the uncertainty around the performance difference.

In [ ]:
# ============================================================
# 5. Paired bootstrap comparison
# ============================================================

rng = np.random.default_rng(
    BOOTSTRAP_RANDOM_SEED
)

participant_count = len(
    paired_predictions
)


bootstrap_storage = {
    (
        output_name,
        metric_name,
    ): {
        "original": [],
        "prebaseline": [],
        "difference": [],
    }
    for output_name in OUTPUT_COLUMNS
    for metric_name in METRIC_NAMES
}


completed_iterations = 0


while completed_iterations < BOOTSTRAP_ITERATIONS:

    sampled_indices = rng.integers(
        low=0,
        high=participant_count,
        size=participant_count,
    )


    sampled_targets = targets[
        sampled_indices
    ]


    if np.unique(
        sampled_targets
    ).size < 2:
        continue


    for output_name, columns in (
        OUTPUT_COLUMNS.items()
    ):

        sampled_original = (
            paired_predictions[
                columns[
                    "original"
                ]
            ]
            .to_numpy(
                dtype=float
            )[
                sampled_indices
            ]
        )

        sampled_prebaseline = (
            paired_predictions[
                columns[
                    "prebaseline"
                ]
            ]
            .to_numpy(
                dtype=float
            )[
                sampled_indices
            ]
        )


        for metric_name in METRIC_NAMES:

            original_value = calculate_metric(
                sampled_targets,
                sampled_original,
                metric_name,
            )

            prebaseline_value = calculate_metric(
                sampled_targets,
                sampled_prebaseline,
                metric_name,
            )

            difference = (
                prebaseline_value
                - original_value
            )


            bootstrap_storage[
                (
                    output_name,
                    metric_name,
                )
            ][
                "original"
            ].append(
                original_value
            )

            bootstrap_storage[
                (
                    output_name,
                    metric_name,
                )
            ][
                "prebaseline"
            ].append(
                prebaseline_value
            )

            bootstrap_storage[
                (
                    output_name,
                    metric_name,
                )
            ][
                "difference"
            ].append(
                difference
            )


    completed_iterations += 1


bootstrap_rows = []


for output_name, columns in (
    OUTPUT_COLUMNS.items()
):

    original_full = paired_predictions[
        columns[
            "original"
        ]
    ].to_numpy(
        dtype=float
    )

    prebaseline_full = paired_predictions[
        columns[
            "prebaseline"
        ]
    ].to_numpy(
        dtype=float
    )


    for metric_name in METRIC_NAMES:

        original_point = calculate_metric(
            targets,
            original_full,
            metric_name,
        )

        prebaseline_point = calculate_metric(
            targets,
            prebaseline_full,
            metric_name,
        )

        point_difference = (
            prebaseline_point
            - original_point
        )


        difference_distribution = np.asarray(
            bootstrap_storage[
                (
                    output_name,
                    metric_name,
                )
            ][
                "difference"
            ],
            dtype=float,
        )


        ci_lower, ci_upper = np.percentile(
            difference_distribution,
            [
                2.5,
                97.5,
            ],
        )


        probability_nonpositive = np.mean(
            difference_distribution
            <= 0.0
        )

        probability_nonnegative = np.mean(
            difference_distribution
            >= 0.0
        )

        descriptive_p = min(
            1.0,
            2.0
            * min(
                probability_nonpositive,
                probability_nonnegative,
            ),
        )


        bootstrap_rows.append(
            {
                "OUTPUT":
                    output_name,

                "METRIC":
                    metric_name,

                "ORIGINAL_POINT":
                    original_point,

                "PREBASELINE_POINT":
                    prebaseline_point,

                "PREBASELINE_MINUS_ORIGINAL":
                    point_difference,

                "DIFFERENCE_CI_95_LOWER":
                    float(
                        ci_lower
                    ),

                "DIFFERENCE_CI_95_UPPER":
                    float(
                        ci_upper
                    ),

                "DESCRIPTIVE_TWO_SIDED_BOOTSTRAP_P":
                    float(
                        descriptive_p
                    ),

                "CI_EXCLUDES_ZERO":
                    bool(
                        (
                            ci_lower > 0.0
                        )
                        or
                        (
                            ci_upper < 0.0
                        )
                    ),
            }
        )


paired_bootstrap_results = pd.DataFrame(
    bootstrap_rows
)


PAIRED_BOOTSTRAP_RESULTS_PATH = (
    TABLE_DIR
    / "paired_bootstrap_metric_comparison.csv"
)

paired_bootstrap_results.to_csv(
    PAIRED_BOOTSTRAP_RESULTS_PATH,
    index=False,
)


display(
    paired_bootstrap_results.round(
        6
    )
)

## 1.6. Primary Hybrid result

In [ ]:
# ============================================================
# 6. Primary Hybrid result
# ============================================================

primary_hybrid_comparison = (
    paired_bootstrap_results.loc[
        paired_bootstrap_results[
            "OUTPUT"
        ]
        == "Hybrid"
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


PRIMARY_HYBRID_PATH = (
    TABLE_DIR
    / "primary_hybrid_original_vs_prebaseline.csv"
)

primary_hybrid_comparison.to_csv(
    PRIMARY_HYBRID_PATH,
    index=False,
)


print("=" * 72)
print("PRIMARY HYBRID COMPARISON")
print("=" * 72)

display(
    primary_hybrid_comparison.round(
        6
    )
)

## 1.7. Participant-level prediction and uncertainty changes

In [ ]:
# ============================================================
# 7. Participant-level prediction and uncertainty changes
# ============================================================

paired_predictions[
    "HYBRID_PROBABILITY_CHANGE"
] = (
    paired_predictions[
        "FINAL_P_pMCI_PREBASELINE"
    ]
    -
    paired_predictions[
        "FINAL_P_pMCI_ORIGINAL"
    ]
)


paired_predictions[
    "ABS_HYBRID_PROBABILITY_CHANGE"
] = paired_predictions[
    "HYBRID_PROBABILITY_CHANGE"
].abs()


paired_predictions[
    "ORIGINAL_CLASS"
] = (
    paired_predictions[
        "FINAL_P_pMCI_ORIGINAL"
    ]
    >= CLASSIFICATION_THRESHOLD
).astype(
    int
)


paired_predictions[
    "PREBASELINE_CLASS"
] = (
    paired_predictions[
        "FINAL_P_pMCI_PREBASELINE"
    ]
    >= CLASSIFICATION_THRESHOLD
).astype(
    int
)


paired_predictions[
    "CLASS_CHANGED"
] = (
    paired_predictions[
        "ORIGINAL_CLASS"
    ]
    !=
    paired_predictions[
        "PREBASELINE_CLASS"
    ]
)


paired_predictions[
    "ORIGINAL_CORRECT"
] = (
    paired_predictions[
        "ORIGINAL_CLASS"
    ]
    ==
    paired_predictions[
        "TARGET"
    ]
)


paired_predictions[
    "PREBASELINE_CORRECT"
] = (
    paired_predictions[
        "PREBASELINE_CLASS"
    ]
    ==
    paired_predictions[
        "TARGET"
    ]
)


paired_predictions[
    "CORRECTNESS_CHANGE"
] = np.select(
    [
        (
            paired_predictions[
                "ORIGINAL_CORRECT"
            ]
            &
            ~paired_predictions[
                "PREBASELINE_CORRECT"
            ]
        ),

        (
            ~paired_predictions[
                "ORIGINAL_CORRECT"
            ]
            &
            paired_predictions[
                "PREBASELINE_CORRECT"
            ]
        ),
    ],
    [
        "Became incorrect",
        "Became correct",
    ],
    default="Unchanged",
)


paired_predictions[
    "UNCERTAINTY_CHANGE"
] = (
    paired_predictions[
        "FINAL_UNCERTAINTY_PREBASELINE"
    ]
    -
    paired_predictions[
        "FINAL_UNCERTAINTY_ORIGINAL"
    ]
)


prediction_change_summary = pd.DataFrame(
    [
        {
            "PARTICIPANTS":
                len(
                    paired_predictions
                ),

            "PEARSON_PROBABILITY_CORRELATION":
                float(
                    np.corrcoef(
                        paired_predictions[
                            "FINAL_P_pMCI_ORIGINAL"
                        ],
                        paired_predictions[
                            "FINAL_P_pMCI_PREBASELINE"
                        ],
                    )[
                        0,
                        1,
                    ]
                ),

            "MEAN_SIGNED_PROBABILITY_CHANGE":
                float(
                    paired_predictions[
                        "HYBRID_PROBABILITY_CHANGE"
                    ].mean()
                ),

            "MEAN_ABSOLUTE_PROBABILITY_CHANGE":
                float(
                    paired_predictions[
                        "ABS_HYBRID_PROBABILITY_CHANGE"
                    ].mean()
                ),

            "MEDIAN_ABSOLUTE_PROBABILITY_CHANGE":
                float(
                    paired_predictions[
                        "ABS_HYBRID_PROBABILITY_CHANGE"
                    ].median()
                ),

            "MAXIMUM_ABSOLUTE_PROBABILITY_CHANGE":
                float(
                    paired_predictions[
                        "ABS_HYBRID_PROBABILITY_CHANGE"
                    ].max()
                ),

            "CLASS_CHANGED_AT_0_50":
                int(
                    paired_predictions[
                        "CLASS_CHANGED"
                    ].sum()
                ),

            "BECAME_CORRECT":
                int(
                    (
                        paired_predictions[
                            "CORRECTNESS_CHANGE"
                        ]
                        == "Became correct"
                    ).sum()
                ),

            "BECAME_INCORRECT":
                int(
                    (
                        paired_predictions[
                            "CORRECTNESS_CHANGE"
                        ]
                        == "Became incorrect"
                    ).sum()
                ),

            "MEAN_UNCERTAINTY_CHANGE":
                float(
                    paired_predictions[
                        "UNCERTAINTY_CHANGE"
                    ].mean()
                ),
        }
    ]
)


PREDICTION_CHANGE_SUMMARY_PATH = (
    TABLE_DIR
    / "participant_prediction_change_summary.csv"
)

prediction_change_summary.to_csv(
    PREDICTION_CHANGE_SUMMARY_PATH,
    index=False,
)


display(
    prediction_change_summary.round(
        6
    )
)


display(
    paired_predictions[
        "CORRECTNESS_CHANGE"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "CORRECTNESS_CHANGE"
    )
    .reset_index(
        name="PARTICIPANTS"
    )
)

## 1.8. Load original and pre-baseline modality availability

In [ ]:
# ============================================================
# 8. Load original and pre-baseline modality availability
# ============================================================

def load_out_of_fold_availability(
    input_root,
    suffix,
):
    fold_tables = []


    for fold in EXPECTED_FOLDS:

        fold_path = (
            input_root
            / (
                "mci_prognosis_outer_fold_"
                f"{fold}_final_task_ready.csv"
            )
        )


        required_columns = [
            "RID",
            "OUTER_FOLD",
            "DATA_ROLE",
            *BRANCH_MASK_COLUMNS,
        ]


        fold_table = pd.read_csv(
            fold_path,
            usecols=required_columns,
        )


        fold_table[
            "RID"
        ] = pd.to_numeric(
            fold_table[
                "RID"
            ],
            errors="raise",
        ).astype(
            int
        )


        fold_test = (
            fold_table.loc[
                (
                    pd.to_numeric(
                        fold_table[
                            "OUTER_FOLD"
                        ],
                        errors="raise",
                    ).astype(
                        int
                    )
                    == fold
                )
                &
                (
                    fold_table[
                        "DATA_ROLE"
                    ]
                    .astype(
                        str
                    )
                    .str.strip()
                    .str.lower()
                    == "test"
                )
            ]
            .copy()
        )


        fold_test[
            "FOLD"
        ] = fold


        fold_tables.append(
            fold_test[
                [
                    "FOLD",
                    "RID",
                    *BRANCH_MASK_COLUMNS,
                ]
            ]
        )


    availability = pd.concat(
        fold_tables,
        ignore_index=True,
    )


    if len(
        availability
    ) != EXPECTED_PARTICIPANTS:
        raise ValueError(
            f"{suffix}: expected "
            f"{EXPECTED_PARTICIPANTS} availability rows, "
            f"found {len(availability)}."
        )


    rename_map = {
        column:
            f"{column}_{suffix}"
        for column in BRANCH_MASK_COLUMNS
    }


    return availability.rename(
        columns=rename_map
    )


original_availability = (
    load_out_of_fold_availability(
        ORIGINAL_INPUT_ROOT,
        "ORIGINAL",
    )
)

prebaseline_availability = (
    load_out_of_fold_availability(
        PREBASELINE_INPUT_ROOT,
        "PREBASELINE",
    )
)


paired_availability = (
    original_availability.merge(
        prebaseline_availability,
        on=[
            "FOLD",
            "RID",
        ],
        how="inner",
        validate="one_to_one",
    )
)


paired_predictions = (
    paired_predictions.merge(
        paired_availability,
        on=[
            "FOLD",
            "RID",
        ],
        how="left",
        validate="one_to_one",
    )
)


if paired_predictions[
    [
        f"{column}_ORIGINAL"
        for column in BRANCH_MASK_COLUMNS
    ]
    +
    [
        f"{column}_PREBASELINE"
        for column in BRANCH_MASK_COLUMNS
    ]
].isna().any().any():
    raise ValueError(
        "At least one participant is missing modality "
        "availability information."
    )


for column in BRANCH_MASK_COLUMNS:

    original_column = (
        f"{column}_ORIGINAL"
    )

    prebaseline_column = (
        f"{column}_PREBASELINE"
    )

    change_column = (
        column.replace(
            "BRANCH_MASK__",
            "LOST__",
        )
    )


    paired_predictions[
        change_column
    ] = (
        (
            paired_predictions[
                original_column
            ]
            == 1
        )
        &
        (
            paired_predictions[
                prebaseline_column
            ]
            == 0
        )
    ).astype(
        int
    )


LOST_MODALITY_COLUMNS = [
    column.replace(
        "BRANCH_MASK__",
        "LOST__",
    )
    for column in BRANCH_MASK_COLUMNS
]


paired_predictions[
    "LOST_MODALITY_COUNT"
] = paired_predictions[
    LOST_MODALITY_COLUMNS
].sum(
    axis=1
)


paired_predictions[
    "AVAILABILITY_CHANGED"
] = (
    paired_predictions[
        "LOST_MODALITY_COUNT"
    ]
    > 0
)


paired_predictions[
    "LOST_MODALITIES"
] = paired_predictions.apply(
    lambda row: (
        "None"
        if row[
            "LOST_MODALITY_COUNT"
        ]
        == 0
        else
        " + ".join(
            BRANCH_DISPLAY_NAMES[
                branch_column
            ]
            for branch_column, lost_column
            in zip(
                BRANCH_MASK_COLUMNS,
                LOST_MODALITY_COLUMNS,
            )
            if row[
                lost_column
            ]
            == 1
        )
    ),
    axis=1,
)


paired_predictions.to_csv(
    ALIGNED_PREDICTIONS_PATH,
    index=False,
)


availability_change_summary = pd.DataFrame(
    [
        {
            "PARTICIPANTS":
                len(
                    paired_predictions
                ),

            "AVAILABILITY_UNCHANGED":
                int(
                    (
                        ~paired_predictions[
                            "AVAILABILITY_CHANGED"
                        ]
                    ).sum()
                ),

            "LOST_AT_LEAST_ONE_MODALITY":
                int(
                    paired_predictions[
                        "AVAILABILITY_CHANGED"
                    ].sum()
                ),

            "LOST_ONE_MODALITY":
                int(
                    (
                        paired_predictions[
                            "LOST_MODALITY_COUNT"
                        ]
                        == 1
                    ).sum()
                ),

            "LOST_TWO_OR_MORE_MODALITIES":
                int(
                    (
                        paired_predictions[
                            "LOST_MODALITY_COUNT"
                        ]
                        >= 2
                    ).sum()
                ),
        }
    ]
)


display(
    availability_change_summary
)


display(
    paired_predictions[
        "LOST_MODALITIES"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "LOST_MODALITIES"
    )
    .reset_index(
        name="PARTICIPANTS"
    )
)

## 1.9. Prediction changes by availability-change group

This section separates participants whose branch availability stayed unchanged from participants who lost at least one branch in the pre-baseline experiment.

In [ ]:
# ============================================================
# 9. Prediction changes by availability-change group
# ============================================================

availability_group_rows = []


for group_name, group_table in [
    (
        "Availability unchanged",
        paired_predictions.loc[
            ~paired_predictions[
                "AVAILABILITY_CHANGED"
            ]
        ],
    ),

    (
        "Lost at least one modality",
        paired_predictions.loc[
            paired_predictions[
                "AVAILABILITY_CHANGED"
            ]
        ],
    ),
]:

    group_targets = group_table[
        "TARGET"
    ].to_numpy(
        dtype=int
    )

    original_probabilities = group_table[
        "FINAL_P_pMCI_ORIGINAL"
    ].to_numpy(
        dtype=float
    )

    prebaseline_probabilities = group_table[
        "FINAL_P_pMCI_PREBASELINE"
    ].to_numpy(
        dtype=float
    )


    row = {
        "GROUP":
            group_name,

        "PARTICIPANTS":
            len(
                group_table
            ),

        "MEAN_ABSOLUTE_PROBABILITY_CHANGE":
            float(
                group_table[
                    "ABS_HYBRID_PROBABILITY_CHANGE"
                ].mean()
            ),

        "MEDIAN_ABSOLUTE_PROBABILITY_CHANGE":
            float(
                group_table[
                    "ABS_HYBRID_PROBABILITY_CHANGE"
                ].median()
            ),

        "CLASS_CHANGED":
            int(
                group_table[
                    "CLASS_CHANGED"
                ].sum()
            ),

        "BECAME_CORRECT":
            int(
                (
                    group_table[
                        "CORRECTNESS_CHANGE"
                    ]
                    == "Became correct"
                ).sum()
            ),

        "BECAME_INCORRECT":
            int(
                (
                    group_table[
                        "CORRECTNESS_CHANGE"
                    ]
                    == "Became incorrect"
                ).sum()
            ),

        "MEAN_UNCERTAINTY_CHANGE":
            float(
                group_table[
                    "UNCERTAINTY_CHANGE"
                ].mean()
            ),
    }


    if (
        len(
            group_table
        )
        > 1
        and
        np.unique(
            group_targets
        ).size
        == 2
    ):

        row[
            "ORIGINAL_ROC_AUC"
        ] = float(
            roc_auc_score(
                group_targets,
                original_probabilities,
            )
        )

        row[
            "PREBASELINE_ROC_AUC"
        ] = float(
            roc_auc_score(
                group_targets,
                prebaseline_probabilities,
            )
        )

        row[
            "AUC_CHANGE"
        ] = (
            row[
                "PREBASELINE_ROC_AUC"
            ]
            -
            row[
                "ORIGINAL_ROC_AUC"
            ]
        )

    else:
        row[
            "ORIGINAL_ROC_AUC"
        ] = np.nan

        row[
            "PREBASELINE_ROC_AUC"
        ] = np.nan

        row[
            "AUC_CHANGE"
        ] = np.nan


    availability_group_rows.append(
        row
    )


availability_group_comparison = pd.DataFrame(
    availability_group_rows
)


AVAILABILITY_GROUP_COMPARISON_PATH = (
    TABLE_DIR
    / "availability_change_group_comparison.csv"
)

availability_group_comparison.to_csv(
    AVAILABILITY_GROUP_COMPARISON_PATH,
    index=False,
)


display(
    availability_group_comparison.round(
        6
    )
)

## 1.10. Changes by lost-modality pattern

In [ ]:
# ============================================================
# 10. Changes by lost-modality pattern
# ============================================================

lost_pattern_rows = []


for lost_pattern, group_table in (
    paired_predictions.groupby(
        "LOST_MODALITIES",
        dropna=False,
    )
):

    group_targets = group_table[
        "TARGET"
    ].to_numpy(
        dtype=int
    )


    row = {
        "LOST_MODALITIES":
            lost_pattern,

        "PARTICIPANTS":
            len(
                group_table
            ),

        "pMCI_PARTICIPANTS":
            int(
                group_table[
                    "TARGET"
                ].sum()
            ),

        "MEAN_ABSOLUTE_PROBABILITY_CHANGE":
            float(
                group_table[
                    "ABS_HYBRID_PROBABILITY_CHANGE"
                ].mean()
            ),

        "MEDIAN_ABSOLUTE_PROBABILITY_CHANGE":
            float(
                group_table[
                    "ABS_HYBRID_PROBABILITY_CHANGE"
                ].median()
            ),

        "MEAN_SIGNED_PROBABILITY_CHANGE":
            float(
                group_table[
                    "HYBRID_PROBABILITY_CHANGE"
                ].mean()
            ),

        "CLASS_CHANGED":
            int(
                group_table[
                    "CLASS_CHANGED"
                ].sum()
            ),

        "BECAME_CORRECT":
            int(
                (
                    group_table[
                        "CORRECTNESS_CHANGE"
                    ]
                    == "Became correct"
                ).sum()
            ),

        "BECAME_INCORRECT":
            int(
                (
                    group_table[
                        "CORRECTNESS_CHANGE"
                    ]
                    == "Became incorrect"
                ).sum()
            ),

        "MEAN_UNCERTAINTY_CHANGE":
            float(
                group_table[
                    "UNCERTAINTY_CHANGE"
                ].mean()
            ),
    }


    if (
        len(
            group_table
        )
        >= 10
        and
        np.unique(
            group_targets
        ).size
        == 2
    ):

        original_auc = roc_auc_score(
            group_targets,
            group_table[
                "FINAL_P_pMCI_ORIGINAL"
            ],
        )

        prebaseline_auc = roc_auc_score(
            group_targets,
            group_table[
                "FINAL_P_pMCI_PREBASELINE"
            ],
        )

        row[
            "ORIGINAL_ROC_AUC"
        ] = float(
            original_auc
        )

        row[
            "PREBASELINE_ROC_AUC"
        ] = float(
            prebaseline_auc
        )

        row[
            "AUC_CHANGE"
        ] = float(
            prebaseline_auc
            - original_auc
        )

    else:
        row[
            "ORIGINAL_ROC_AUC"
        ] = np.nan

        row[
            "PREBASELINE_ROC_AUC"
        ] = np.nan

        row[
            "AUC_CHANGE"
        ] = np.nan


    lost_pattern_rows.append(
        row
    )


lost_modality_pattern_comparison = (
    pd.DataFrame(
        lost_pattern_rows
    )
    .sort_values(
        [
            "PARTICIPANTS",
            "LOST_MODALITIES",
        ],
        ascending=[
            False,
            True,
        ],
    )
    .reset_index(
        drop=True
    )
)


LOST_PATTERN_PATH = (
    TABLE_DIR
    / "lost_modality_pattern_comparison.csv"
)

lost_modality_pattern_comparison.to_csv(
    LOST_PATTERN_PATH,
    index=False,
)


display(
    lost_modality_pattern_comparison.round(
        6
    )
)

## 1.11. Fold-level comparison

In [ ]:
# ============================================================
# 11. Fold-level comparison
# ============================================================

fold_rows = []


for fold in EXPECTED_FOLDS:

    fold_table = paired_predictions.loc[
        paired_predictions[
            "FOLD"
        ]
        == fold
    ]

    fold_targets = fold_table[
        "TARGET"
    ].to_numpy(
        dtype=int
    )

    original_probabilities = fold_table[
        "FINAL_P_pMCI_ORIGINAL"
    ].to_numpy(
        dtype=float
    )

    prebaseline_probabilities = fold_table[
        "FINAL_P_pMCI_PREBASELINE"
    ].to_numpy(
        dtype=float
    )


    for metric_name in METRIC_NAMES:

        original_value = calculate_metric(
            fold_targets,
            original_probabilities,
            metric_name,
        )

        prebaseline_value = calculate_metric(
            fold_targets,
            prebaseline_probabilities,
            metric_name,
        )


        fold_rows.append(
            {
                "FOLD":
                    fold,

                "METRIC":
                    metric_name,

                "ORIGINAL":
                    original_value,

                "PREBASELINE":
                    prebaseline_value,

                "PREBASELINE_MINUS_ORIGINAL":
                    (
                        prebaseline_value
                        - original_value
                    ),
            }
        )


fold_level_comparison = pd.DataFrame(
    fold_rows
)


FOLD_LEVEL_PATH = (
    TABLE_DIR
    / "fold_level_original_vs_prebaseline.csv"
)

fold_level_comparison.to_csv(
    FOLD_LEVEL_PATH,
    index=False,
)


display(
    fold_level_comparison.round(
        6
    )
)

## 1.12. ROC and precision-recall curves

In [ ]:
# ============================================================
# 12. ROC and precision-recall curves
# ============================================================

original_hybrid = paired_predictions[
    "FINAL_P_pMCI_ORIGINAL"
].to_numpy(
    dtype=float
)

prebaseline_hybrid = paired_predictions[
    "FINAL_P_pMCI_PREBASELINE"
].to_numpy(
    dtype=float
)


original_auc = roc_auc_score(
    targets,
    original_hybrid,
)

prebaseline_auc = roc_auc_score(
    targets,
    prebaseline_hybrid,
)


original_fpr, original_tpr, _ = (
    roc_curve(
        targets,
        original_hybrid,
    )
)

prebaseline_fpr, prebaseline_tpr, _ = (
    roc_curve(
        targets,
        prebaseline_hybrid,
    )
)


plt.figure(
    figsize=(
        7,
        6,
    )
)

plt.plot(
    original_fpr,
    original_tpr,
    label=(
        "Original fixed fusion "
        f"(AUC={original_auc:.4f})"
    ),
)

plt.plot(
    prebaseline_fpr,
    prebaseline_tpr,
    label=(
        "Pre-baseline only "
        f"(AUC={prebaseline_auc:.4f})"
    ),
)

plt.plot(
    [
        0,
        1,
    ],
    [
        0,
        1,
    ],
    linestyle="--",
    label="Chance",
)

plt.xlabel(
    "False positive rate"
)

plt.ylabel(
    "True positive rate"
)

plt.title(
    "Hybrid ROC curves"
)

plt.legend()

plt.tight_layout()


ROC_PATH = (
    FIGURE_DIR
    / "hybrid_roc_original_vs_prebaseline.png"
)

plt.savefig(
    ROC_PATH,
    dpi=300,
    bbox_inches="tight",
)

plt.show()


original_ap = average_precision_score(
    targets,
    original_hybrid,
)

prebaseline_ap = average_precision_score(
    targets,
    prebaseline_hybrid,
)


original_precision, original_recall, _ = (
    precision_recall_curve(
        targets,
        original_hybrid,
    )
)

prebaseline_precision, prebaseline_recall, _ = (
    precision_recall_curve(
        targets,
        prebaseline_hybrid,
    )
)


plt.figure(
    figsize=(
        7,
        6,
    )
)

plt.plot(
    original_recall,
    original_precision,
    label=(
        "Original fixed fusion "
        f"(AP={original_ap:.4f})"
    ),
)

plt.plot(
    prebaseline_recall,
    prebaseline_precision,
    label=(
        "Pre-baseline only "
        f"(AP={prebaseline_ap:.4f})"
    ),
)

plt.xlabel(
    "Recall"
)

plt.ylabel(
    "Precision"
)

plt.title(
    "Hybrid precision-recall curves"
)

plt.legend()

plt.tight_layout()


PR_PATH = (
    FIGURE_DIR
    / "hybrid_precision_recall_original_vs_prebaseline.png"
)

plt.savefig(
    PR_PATH,
    dpi=300,
    bbox_inches="tight",
)

plt.show()

## 1.13. Final comparison summary

In [ ]:
# ============================================================
# 13. Final comparison summary
# ============================================================

print("=" * 72)
print("FINAL ORIGINAL VERSUS PRE-BASELINE COMPARISON")
print("=" * 72)


for _, result in (
    primary_hybrid_comparison.iterrows()
):

    metric_name = result[
        "METRIC"
    ]

    original_value = result[
        "ORIGINAL_POINT"
    ]

    prebaseline_value = result[
        "PREBASELINE_POINT"
    ]

    difference = result[
        "PREBASELINE_MINUS_ORIGINAL"
    ]

    ci_lower = result[
        "DIFFERENCE_CI_95_LOWER"
    ]

    ci_upper = result[
        "DIFFERENCE_CI_95_UPPER"
    ]

    p_value = result[
        "DESCRIPTIVE_TWO_SIDED_BOOTSTRAP_P"
    ]


    print(
        f"\n{metric_name}"
    )

    print(
        f"  Original fixed fusion: "
        f"{original_value:.6f}"
    )

    print(
        f"  Pre-baseline only:     "
        f"{prebaseline_value:.6f}"
    )

    print(
        f"  Pre-baseline minus original: "
        f"{difference:.6f}"
    )

    print(
        "  Paired 95% bootstrap CI: "
        f"[{ci_lower:.6f}, {ci_upper:.6f}]"
    )

    print(
        f"  Descriptive two-sided bootstrap p: "
        f"{p_value:.4f}"
    )


auc_result = (
    primary_hybrid_comparison.loc[
        primary_hybrid_comparison[
            "METRIC"
        ]
        == "ROC_AUC"
    ]
    .iloc[
        0
    ]
)


print(
    "\n" + "-" * 72
)

if bool(
    auc_result[
        "CI_EXCLUDES_ZERO"
    ]
):
    print(
        "The paired ROC AUC interval excludes zero. "
        "The performance change is larger than expected "
        "from participant resampling alone."
    )

else:
    print(
        "The paired ROC AUC interval includes zero. "
        "The observed AUC change is not decisive in this "
        "paired bootstrap analysis."
    )


print(
    "\nInterpret the result together with the availability "
    "tables: this comparison reflects both stricter timing "
    "and the modalities made unavailable by that rule."
)


print(
    f"\nPrimary result table:\n"
    f"{PRIMARY_HYBRID_PATH}"
)

print(
    f"\nComplete comparison folder:\n"
    f"{COMPARISON_ROOT}"
)